In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sns
import copy
import pickle
import shap
from models import *
from utilities import *
import warnings
warnings.filterwarnings('ignore')

2026-08-06 13:48:09.522701: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-06 13:48:09.565093: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-06 13:48:09.565129: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-06 13:48:09.566344: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-06 13:48:09.574010: I tensorflow/core/platform/cpu_feature_guar

[INFO] No GPU found, using CPU.


2026-08-06 13:48:11.424284: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [3]:
##########################################################################################################

In [4]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("Eager mode:", tf.executing_eagerly())
print("GPUs:", tf.config.list_physical_devices('GPU'))

TF version: 2.15.0
Eager mode: True
GPUs: []


In [5]:
# shap.explainers._deep.deep_tf.op_handlers["AddV2"] = shap.explainers._deep.deep_tf.passthrough 
# shap.explainers._deep.deep_tf.op_handlers["FusedBatchNormV3"] = shap.explainers._deep.deep_tf.passthrough

In [6]:
################### data import ###############

In [7]:
data_dir = '../results/data/'
data_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_dy_filter.pkl')
ba_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_ba_filter.pkl')
data_copy['Id_encoded'], uniques = pd.factorize(data_copy['SCCRIP_ID'])
data_copy = data_copy.fillna(0)
mapping_dict = {value: index for index, value in enumerate(uniques)}

In [8]:
hu = read_sas('../data/Apply_MI_Zheng_2023_01/hu.sas7bdat')

In [9]:
bmt = read_sas('../data/Apply_MI_Zheng_2023_01/othertreatment_summary.sas7bdat')
bmt = bmt[['SCCRIP_ID', 'BMT']]
bmt_ids = set(bmt[bmt['BMT'] == 1].SCCRIP_ID)
data_copy_filter = data_copy[~data_copy["SCCRIP_ID"].isin(bmt_ids)]
len(data_copy_filter.SCCRIP_ID.unique())

1264

In [10]:
data_copy_nohu_f = data_copy_filter[data_copy_filter['HU_status']==0]
data_copy_hu_f = data_copy_filter[data_copy_filter['HU_status']!=0]

In [11]:
data_copy_nohu = data_copy[data_copy['HU_status']==0]
data_copy_hu = data_copy[data_copy['HU_status']!=0]

In [12]:
len(set(data_copy_nohu['SCCRIP_ID']).intersection(bmt_ids)), len(set(data_copy_hu['SCCRIP_ID']).intersection(bmt_ids))

(3, 3)

In [13]:
ba_copy['Id_encoded'] = ba_copy['SCCRIP_ID'].map(mapping_dict)
ba_copy = ba_copy[ba_copy['Id_encoded'].notna()]
ba_copy = ba_copy.drop_duplicates(subset='SCCRIP_ID', keep='last')
ba_copy.drop('SCCRIP_ID', inplace=True, axis=1)
ba_copy = ba_copy.fillna(0)

In [14]:
feature_dy = data_copy_nohu_f.drop(['HU_status','Id_encoded','SCCRIP_ID', 'date'],axis=1).columns
feature_ba = ba_copy.drop(['Id_encoded'], axis=1).columns

In [15]:
################ data process ##################

In [16]:
seq_length = 5

In [29]:
####################################################

In [ ]:
r2 = []
shap_impact = []
sequences, seq_ba, labels, date,  y_target = create_dataset(data_copy_hu_f, ba_copy, seq_length, mapping_dict)
sequences_copy_h, seq_ba_copy, labels_copy, date_copy, y_target_copy = shuffle_dataset(sequences, seq_ba, labels, date, y_target)
X, y, X_bg, dt, ls, seq_map = create_sequences_with_background(sequences_copy_h, labels_copy, seq_ba_copy, date_copy,y_target_copy, seq_length)
X_normalized, X_bg_normalized = normalize_X(X, X_bg)

while len(shap_impact) < 20:
    print(len(shap_impact))
    
    sequences, seq_ba, labels, date,  y_target = create_dataset(data_copy_hu_f, ba_copy, seq_length, mapping_dict)
    sequences_copy_h, seq_ba_copy, labels_copy, date_copy, y_target_copy = shuffle_dataset(sequences, seq_ba, labels, date, y_target)
    X, y, X_bg, dt, ls, seq_map = create_sequences_with_background(sequences_copy_h, labels_copy, seq_ba_copy, date_copy,y_target_copy, seq_length)
    X_normalized, X_bg_normalized = normalize_X(X, X_bg)
    
    (X_train_hu,X_val_hu,X_test_hu,
                X_bg_train_hu,X_bg_val_hu,X_bg_test_hu,
                dt_train_hu, dt_val_hu, dt_test_hu, 
                y_train_hu,y_val_hu,y_test_hu,
                ls_train_hu, ls_val_hu, ls_test_hu,
                seq_map_train_hu, seq_map_val_hu, seq_map_test_hu) = split_dataset(X_normalized, X_bg_normalized,dt, y, ls, seq_map)
    
    combined_model_l = CombinedLSTMModel(seq_length=seq_length, n_features=int(X_train_hu.shape[2]), bg_features=int(X_bg_train_hu.shape[2]), summary=False)
    
    history = combined_model_l.train(
    X_train_hu, X_bg_train_hu, dt_train_hu, y_train_hu,
    X_val_hu,   X_bg_val_hu,   dt_val_hu,   y_val_hu,
    epochs=20, batch_size=64,
    checkpoint_path="../results/models/T_best2_loop.h5")
    y_pred_hu = combined_model_l.predict(X_test_hu, X_bg_test_hu, dt_test_hu)
    r2_hu = r2_score(y_test_hu.flatten(), y_pred_hu.flatten())
    if r2_hu > 0:
        r2.append(r2_hu)

        explainer = shap.GradientExplainer(combined_model_l.model, [X_test_hu, X_bg_test_hu, dt_test_hu])
        shap_values = explainer.shap_values([X_test_hu, X_bg_test_hu, dt_test_hu])

        shap_val_d = []
        train_d = []
        
        for i in range (5):
            shap_val_d.append(pd.DataFrame(shap_values[0][:,i,:,0], columns=feature_dy))
            train_d.append(pd.DataFrame(X_test_hu[:,i,:],columns=feature_dy))
            
        shap_val_b = pd.DataFrame(shap_values[1][:,0,:,0], columns=feature_ba)
        train_b = pd.DataFrame(X_bg_test_hu[:,0,:], columns=feature_ba)

        impact_dy = []

        for i in range(5):
            im = ABS_SHAP(shap_val_d[i], train_d[i])
            impact_dy.append(im)
            
        impact_ba = ABS_SHAP(shap_val_b, train_b)
        shap_impact.append((impact_dy, impact_ba))

0
Epoch 1/20
200/208 [===========================>..] - ETA: 0s - loss: 377.2913 - mean_absolute_error: 16.6571
Epoch 1: val_loss improved from inf to 338.81720, saving model to ../results/models/T_best2_loop.h5
208/208 [==============================] - 3s 7ms/step - loss: 376.5608 - mean_absolute_error: 16.6410 - val_loss: 338.8172 - val_mean_absolute_error: 15.1191
Epoch 2/20
201/208 [===========================>..] - ETA: 0s - loss: 376.4004 - mean_absolute_error: 16.6345
Epoch 2: val_loss did not improve from 338.81720
208/208 [==============================] - 1s 5ms/step - loss: 376.5609 - mean_absolute_error: 16.6410 - val_loss: 338.8172 - val_mean_absolute_error: 15.1191
Epoch 3/20
201/208 [===========================>..] - ETA: 0s - loss: 377.7920 - mean_absolute_error: 16.6709
Epoch 3: val_loss did not improve from 338.81720
208/208 [==============================] - 1s 5ms/step - loss: 376.5608 - mean_absolute_error: 16.6410 - val_loss: 338.8172 - val_mean_absolute_error: 1

In [ ]:
with open('../results/data/tmodel/shap_impact_list_noBMTv4.pkl','wb') as f:
    pickle.dump(shap_impact,f)  

In [ ]:
with open('../results/data/tmodel/r2_Tmodel_noBMT.pkl','wb') as f:
    pickle.dump(r2,f)

In [ ]:
#########################################